## Parameters

In [26]:
%%writefile parameters.py
"""
Parameters for Rusanov scheme.
"""

import numpy as np

class Parameters:
    def __init__(self):
        self.gamma = 1.4
        self.R = 287.0

        # Good resolution
        self.Nx = 400
        self.Ny = 5  # Minimal y-direction for 1D Sod
        self.Lx = 10.0
        self.Ly = 0.1

        # Conservative CFL for Rusanov
        self.CFL = 0.3
        self.t_final = 0.5
        self.t_print = 0.05

        self.output_dir = "output"
        self.update_grid()

    def update_grid(self):
        self.dx = self.Lx / (self.Nx - 1)
        self.dy = self.Ly / (self.Ny - 1)

params = Parameters()

Overwriting parameters.py


## Mesh


In [27]:
%%writefile mesh.py
"""
Mesh and grid generation - FDM version with nodes.
"""

import numpy as np
from parameters import params

class Mesh:
    def __init__(self):
        self.Nx = params.Nx
        self.Ny = params.Ny
        self.Lx = params.Lx
        self.Ly = params.Ly
        self.dx = params.dx
        self.dy = params.dy

        # Grid points (nodes) for FDM
        self.x_nodes = np.linspace(0, self.Lx, self.Nx)
        self.y_nodes = np.linspace(0, self.Ly, self.Ny)
        self.X, self.Y = np.meshgrid(self.x_nodes, self.y_nodes, indexing='ij')

        print(f"FDM Grid: {self.Nx} x {self.Ny} nodes")
        print(f"Domain: x∈[0,{self.Lx}], y∈[0,{self.Ly}]")
        print(f"dx={self.dx:.6f}, dy={self.dy:.6f}")

# Global mesh object
mesh = Mesh()

Overwriting mesh.py


## state

In [28]:
%%writefile state.py
"""
Compressible flow state variables at grid nodes.
"""

import numpy as np
from parameters import params
import mesh

class FlowState:
    __slots__ = ['gamma', 'Nx', 'Ny', 'Q', 'rho', 'u', 'v', 'p']

    def __init__(self):
        self.gamma = params.gamma
        self.Nx = mesh.mesh.Nx
        self.Ny = mesh.mesh.Ny

        # Conservative variables Q = [rho, rho*u, rho*v, E] at nodes
        self.Q = np.zeros((4, self.Nx, self.Ny))

        # Primitive variables at nodes
        self.rho = np.ones((self.Nx, self.Ny))
        self.u = np.zeros((self.Nx, self.Ny))
        self.v = np.zeros((self.Nx, self.Ny))
        self.p = np.ones((self.Nx, self.Ny))

    def prim_to_conservative(self):
        """Convert primitive to conservative"""
        self.Q[0] = self.rho
        self.Q[1] = self.rho * self.u
        self.Q[2] = self.rho * self.v
        kinetic = 0.5 * self.rho * (self.u**2 + self.v**2)
        self.Q[3] = self.p / (self.gamma - 1.0) + kinetic

    def conservative_to_primitive(self):
        """Convert conservative to primitive"""
        np.maximum(self.Q[0], 1e-10, out=self.rho)
        self.u = self.Q[1] / self.rho
        self.v = self.Q[2] / self.rho

        kinetic = 0.5 * self.rho * (self.u**2 + self.v**2)
        e_int = self.Q[3] / self.rho - kinetic
        np.maximum(e_int, 1e-10, out=e_int)
        self.p = (self.gamma - 1.0) * self.rho * e_int
        np.maximum(self.p, 1e-10, out=self.p)

    def get_mach_number(self):
        """Compute Mach number"""
        a = np.sqrt(self.gamma * self.p / (self.rho + 1e-14))
        speed = np.sqrt(self.u**2 + self.v**2)
        return speed / (a + 1e-14)

# Global state
state = FlowState()

Overwriting state.py


## Flux

In [29]:
%%writefile flux.py
"""
Physical flux functions for Euler equations - fixed for array operations.
"""

import numpy as np
from parameters import params

def flux_x(Q, gamma=params.gamma):
    """Flux in x-direction"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * rho * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)

    F = np.zeros_like(Q)
    F[0] = Q[1]  # rho*u
    F[1] = Q[1] * u + p
    F[2] = Q[1] * v
    F[3] = u * (Q[3] + p)
    return F

def flux_y(Q, gamma=params.gamma):
    """Flux in y-direction"""
    rho = np.maximum(Q[0], 1e-10)
    u = Q[1] / rho
    v = Q[2] / rho

    kinetic = 0.5 * rho * (u**2 + v**2)
    e_int = Q[3] / rho - kinetic
    e_int = np.maximum(e_int, 1e-10)
    p = (gamma - 1.0) * rho * e_int
    p = np.maximum(p, 1e-10)

    G = np.zeros_like(Q)
    G[0] = Q[2]  # rho*v
    G[1] = Q[2] * u
    G[2] = Q[2] * v + p
    G[3] = v * (Q[3] + p)
    return G

Overwriting flux.py


## WENO5

In [30]:
%%writefile weno5_fdm_fixed.py
"""
Fixed WENO5 scheme for FDM - Stable version for Sod shock tube
"""

import numpy as np
import mesh

def weno5_reconstruct(v, axis):
    """
    WENO5 reconstruction for interface values
    """
    eps = 1e-6

    if axis == 1:  # x-direction
        v0 = v[:, 0:-4, :]
        v1 = v[:, 1:-3, :]
        v2 = v[:, 2:-2, :]
        v3 = v[:, 3:-1, :]
        v4 = v[:, 4:,   :]
    else:  # y-direction
        v0 = v[:, :, 0:-4]
        v1 = v[:, :, 1:-3]
        v2 = v[:, :, 2:-2]
        v3 = v[:, :, 3:-1]
        v4 = v[:, :, 4:]

    # Candidate stencils
    q0 = (2*v2 + 5*v1 - v0) / 6.0   # Left-biased
    q1 = (-v3 + 5*v2 + 2*v1) / 6.0  # Centered
    q2 = (2*v3 + 5*v4 - 11*v2) / 6.0 # Right-biased

    # Smoothness indicators
    beta0 = 13/12*(v0 - 2*v1 + v2)**2 + 1/4*(v0 - 4*v1 + 3*v2)**2
    beta1 = 13/12*(v1 - 2*v2 + v3)**2 + 1/4*(v1 - v3)**2
    beta2 = 13/12*(v2 - 2*v3 + v4)**2 + 1/4*(3*v2 - 4*v3 + v4)**2

    # Weights
    d0, d1, d2 = 0.1, 0.6, 0.3
    alpha0 = d0 / (beta0 + eps)**2
    alpha1 = d1 / (beta1 + eps)**2
    alpha2 = d2 / (beta2 + eps)**2

    return (alpha0*q0 + alpha1*q1 + alpha2*q2) / (alpha0 + alpha1 + alpha2 + eps)

def weno5_flux_x_fixed(Q, gamma=1.4):
    """
    Stable WENO5 flux for FDM using flux vector splitting
    """
    from flux import flux_x

    Nx, Ny = Q.shape[1], Q.shape[2]
    dx = mesh.mesh.dx

    # Compute flux
    F = flux_x(Q, gamma)

    # Global Lax-Friedrichs splitting (more stable than local)
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1]/rho
    v = Q[2]/rho
    p = np.maximum((gamma-1)*(Q[3] - 0.5*rho*(u**2 + v**2)), 1e-8)
    a = np.sqrt(gamma * p / rho)

    # Use GLOBAL maximum wave speed (more stable)
    alpha = np.max(np.abs(u) + a) * 1.1

    # Split fluxes
    F_plus = 0.5 * (F + alpha * Q)
    F_minus = 0.5 * (F - alpha * Q)

    # Add ghost cells
    F_plus_pad = np.pad(F_plus, ((0,0), (3,3), (0,0)), mode='edge')
    F_minus_pad = np.pad(F_minus, ((0,0), (3,3), (0,0)), mode='edge')

    # Reconstruct
    F_plus_recon = weno5_reconstruct(F_plus_pad, axis=1)
    F_minus_recon = weno5_reconstruct(F_minus_pad[:, ::-1, :], axis=1)[:, ::-1, :]

    # Numerical flux at interfaces
    F_interface = F_plus_recon + F_minus_recon

    # Compute derivative
    dFdx = (F_interface[:, 1:Nx+1, :] - F_interface[:, 0:Nx, :]) / dx

    return dFdx

Overwriting weno5_fdm_fixed.py


## Hybrid WENO MUSCL

In [31]:
%%writefile hybrid_weno_muscl.py
"""
Hybrid WENO-MUSCL scheme for FDM
- MUSCL in smooth regions (fast)
- WENO near shocks (accurate)
- Uses shock detector to switch
"""

import numpy as np
import mesh
from parameters import params

def shock_detector(Q, gamma=1.4, threshold=0.5):
    """
    Detect shocks using pressure gradient
    Returns: 1 where WENO needed, 0 where MUSCL is fine
    """
    Nx, Ny = Q.shape[1], Q.shape[2]

    # Get primitive variables
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1] / rho
    v = Q[2] / rho

    # Pressure
    kinetic = 0.5 * rho * (u**2 + v**2)
    p = np.maximum((gamma-1) * (Q[3] - kinetic), 1e-8)

    # Compute pressure gradient indicator
    dpdx = np.zeros((Nx, Ny))
    dpdx[1:-1, :] = np.abs(p[2:, :] - p[:-2, :]) / (2 * mesh.mesh.dx)

    # Normalize by local pressure
    sensor = dpdx / (p + 1e-8)

    # Smooth with neighbors to avoid isolated points
    sensor_smooth = np.zeros_like(sensor)
    for i in range(1, Nx-1):
        sensor_smooth[i, :] = 0.25 * sensor[i-1, :] + 0.5 * sensor[i, :] + 0.25 * sensor[i+1, :]

    # Mark shock cells (1 = use WENO, 0 = use MUSCL)
    shock_cells = (sensor_smooth > threshold).astype(float)

    # Dilate shock region (include neighbors for safety)
    shock_region = np.zeros_like(shock_cells)
    for i in range(1, Nx-1):
        if shock_cells[i, 0] > 0.5:
            shock_region[i-1:i+2, :] = 1.0

    return shock_region

def minmod(a, b, c):
    """Minmod limiter with three arguments"""
    s = (np.sign(a) + np.sign(b) + np.sign(c)) / 3
    return s * np.maximum(0, np.minimum(np.abs(a), np.minimum(np.abs(b), np.abs(c))))

def muscl_flux_x(Q, gamma=1.4):
    """
    MUSCL reconstruction (fast)
    """
    Nx, Ny = Q.shape[1], Q.shape[2]
    dx = mesh.mesh.dx

    # Add ghost cells
    Q_ext = np.zeros((4, Nx+4, Ny))
    Q_ext[:, 2:Nx+2, :] = Q
    Q_ext[:, 0:2, :] = Q[:, 0:1, :]
    Q_ext[:, Nx+2:Nx+4, :] = Q[:, -1:, :]

    dFdx_muscl = np.zeros_like(Q)

    k = 1.0/3.0  # MUSCL parameter

    for i in range(2, Nx+2):
        # Left and right states
        QL = Q_ext[:, i-1, :]
        QR = Q_ext[:, i, :]

        # Compute primitive variables
        rhoL = np.maximum(QL[0], 1e-8)
        rhoR = np.maximum(QR[0], 1e-8)
        uL = QL[1] / rhoL
        uR = QR[1] / rhoR
        pL = (gamma-1) * np.maximum(QL[3] - 0.5*rhoL*uL**2, 1e-8)
        pR = (gamma-1) * np.maximum(QR[3] - 0.5*rhoR*uR**2, 1e-8)

        # Wave speed for Rusanov flux
        aL = np.sqrt(gamma * pL / rhoL)
        aR = np.sqrt(gamma * pR / rhoR)
        alpha = np.maximum(np.abs(uL) + aL, np.abs(uR) + aR)

        # Compute fluxes
        FL = np.zeros_like(QL)
        FL[0] = QL[1]
        FL[1] = QL[1] * uL + pL
        FL[2] = 0.0
        FL[3] = uL * (QL[3] + pL)

        FR = np.zeros_like(QR)
        FR[0] = QR[1]
        FR[1] = QR[1] * uR + pR
        FR[2] = 0.0
        FR[3] = uR * (QR[3] + pR)

        # Rusanov flux
        F_interface = 0.5 * (FL + FR) - 0.5 * alpha * (QR - QL)

        if i-2 >= 0 and i-2 < Nx:
            dFdx_muscl[:, i-2, :] = (F_interface - FL) / dx

    return dFdx_muscl

def weno5_reconstruct(v, axis=1):
    """
    WENO5 reconstruction
    """
    eps = 1e-6

    if axis == 1:
        v0 = v[:, 0:-4, :]
        v1 = v[:, 1:-3, :]
        v2 = v[:, 2:-2, :]
        v3 = v[:, 3:-1, :]
        v4 = v[:, 4:,   :]
    else:
        v0 = v[:, :, 0:-4]
        v1 = v[:, :, 1:-3]
        v2 = v[:, :, 2:-2]
        v3 = v[:, :, 3:-1]
        v4 = v[:, :, 4:]

    # Candidate stencils
    q0 = (2.0*v2 + 5.0*v1 - v0) / 6.0
    q1 = (-v3 + 5.0*v2 + 2.0*v1) / 6.0
    q2 = (2.0*v3 + 5.0*v4 - 11.0*v2) / 6.0

    # Smoothness indicators
    beta0 = 13.0/12.0*(v0 - 2*v1 + v2)**2 + 0.25*(v0 - 4*v1 + 3*v2)**2
    beta1 = 13.0/12.0*(v1 - 2*v2 + v3)**2 + 0.25*(v1 - v3)**2
    beta2 = 13.0/12.0*(v2 - 2*v3 + v4)**2 + 0.25*(3*v2 - 4*v3 + v4)**2

    # Weights
    d0, d1, d2 = 0.1, 0.6, 0.3
    alpha0 = d0 / (beta0 + eps)**2
    alpha1 = d1 / (beta1 + eps)**2
    alpha2 = d2 / (beta2 + eps)**2

    return (alpha0*q0 + alpha1*q1 + alpha2*q2) / (alpha0 + alpha1 + alpha2 + eps)

def weno5_flux_x(Q, gamma=1.4):
    """
    WENO5 flux (accurate but slower)
    """
    Nx, Ny = Q.shape[1], Q.shape[2]
    dx = mesh.mesh.dx

    # Compute flux
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1] / rho
    v = Q[2] / rho
    p = np.maximum((gamma-1)*(Q[3] - 0.5*rho*(u**2 + v**2)), 1e-8)

    F = np.zeros_like(Q)
    F[0] = Q[1]
    F[1] = Q[1] * u + p
    F[2] = Q[2] * u
    F[3] = u * (Q[3] + p)

    # Global wave speed
    a = np.sqrt(gamma * p / rho)
    alpha = np.max(np.abs(u) + a) * 1.1

    # Split fluxes
    F_plus = 0.5 * (F + alpha * Q)
    F_minus = 0.5 * (F - alpha * Q)

    # Add ghost cells
    F_plus_pad = np.pad(F_plus, ((0,0), (3,3), (0,0)), mode='edge')
    F_minus_pad = np.pad(F_minus, ((0,0), (3,3), (0,0)), mode='edge')

    # Reconstruct
    F_plus_recon = weno5_reconstruct(F_plus_pad, axis=1)
    F_minus_recon = weno5_reconstruct(F_minus_pad[:, ::-1, :], axis=1)[:, ::-1, :]

    # Numerical flux
    F_interface = F_plus_recon + F_minus_recon

    # Derivative
    dFdx = (F_interface[:, 1:Nx+1, :] - F_interface[:, 0:Nx, :]) / dx

    return dFdx

def hybrid_weno_muscl_flux_x(Q, gamma=1.4):
    """
    Hybrid scheme: MUSCL everywhere, WENO at shocks
    """
    Nx, Ny = Q.shape[1], Q.shape[2]

    # Detect shocks
    shock_region = shock_detector(Q, gamma, threshold=0.3)

    # Compute both fluxes
    dFdx_muscl = muscl_flux_x(Q, gamma)
    dFdx_weno = weno5_flux_x(Q, gamma)

    # Initialize hybrid flux
    dFdx_hybrid = np.zeros_like(Q)

    # Blend based on shock detector
    for i in range(Nx):
        for j in range(Ny):
            weight_weno = shock_region[i, j]
            weight_muscl = 1.0 - weight_weno

            dFdx_hybrid[:, i, j] = (weight_muscl * dFdx_muscl[:, i, j] +
                                    weight_weno * dFdx_weno[:, i, j])

    return dFdx_hybrid

def compute_rhs_hybrid(Q, dx, dy):
    """
    Compute RHS using hybrid scheme
    """
    dFdx = hybrid_weno_muscl_flux_x(Q, params.gamma)
    dGdy = np.zeros_like(dFdx)
    return -(dFdx + dGdy)

Overwriting hybrid_weno_muscl.py


## MUSCL

In [48]:
%%writefile muscl_fdm_fixed.py
"""
Fixed MUSCL scheme for FDM - Stable and accurate for Sod
"""

import numpy as np
import mesh

def minmod(a, b, c):
    cond = (np.sign(a) == np.sign(b)) & (np.sign(b) == np.sign(c))
    return cond * np.sign(a) * np.minimum(np.abs(a), np.minimum(np.abs(b), np.abs(c)))

def muscl_flux_x(Q, gamma=1.4):
    Nx, Ny = Q.shape[1], Q.shape[2]
    dx = mesh.mesh.dx

    Q_ext = np.zeros((4, Nx+4, Ny))
    Q_ext[:, 2:Nx+2, :] = Q

    # BC
    Q_ext[:, 0:2, :] = Q[:, 0:1, :]
    Q_ext[:, Nx+2:Nx+4, :] = Q[:, -1:, :]

    QL = np.zeros((4, Nx+1, Ny))
    QR = np.zeros((4, Nx+1, Ny))

    for i in range(2, Nx+2):
        for n in range(4):
            dL = Q_ext[n, i] - Q_ext[n, i-1]
            dR = Q_ext[n, i+1] - Q_ext[n, i]

            slope = minmod(dL, dR, 0.5*(dL + dR))

            QL[n, i-2, :] = Q_ext[n, i]     - 0.5 * slope
            QR[n, i-2, :] = Q_ext[n, i+1]   + 0.5 * slope

    dFdx = np.zeros_like(Q)

    for i in range(Nx+1):
        UL = QL[:, i, :]
        UR = QR[:, i, :]

        rhoL = np.maximum(UL[0], 1e-8)
        rhoR = np.maximum(UR[0], 1e-8)

        uL = UL[1] / rhoL
        uR = UR[1] / rhoR

        pL = np.maximum((gamma-1)*(UL[3] - 0.5*rhoL*uL**2), 1e-8)
        pR = np.maximum((gamma-1)*(UR[3] - 0.5*rhoR*uR**2), 1e-8)

        aL = np.sqrt(gamma * pL / rhoL)
        aR = np.sqrt(gamma * pR / rhoR)

        alpha = np.maximum(np.abs(uL)+aL, np.abs(uR)+aR)

        FL = flux_x(UL, gamma)
        FR = flux_x(UR, gamma)

        F = 0.5*(FL + FR) - 0.5*alpha*(UR - UL)

        if i < Nx:
            dFdx[:, i, :] += F
        if i > 0:
            dFdx[:, i-1, :] -= F

    return dFdx / dx

Overwriting muscl_fdm_fixed.py


## Simple TVD

In [33]:
%%writefile tvd_fdm_robust.py
"""
Simple TVD scheme for FDM - Very robust for Sod shock tube
"""

import numpy as np
import mesh

def tvd_flux_x_robust(Q, gamma=1.4):
    """
    Simple TVD scheme with minmod limiter
    Very stable for shock capturing
    """
    from flux import flux_x

    Nx, Ny = Q.shape[1], Q.shape[2]
    dx = mesh.mesh.dx

    # Compute primitive variables
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1] / rho
    p = np.maximum((gamma-1)*(Q[3] - 0.5*rho*u**2), 1e-8)
    a = np.sqrt(gamma * p / rho)

    # Global wave speed
    alpha = np.max(np.abs(u) + a) * 1.1

    # Add ghost cells
    Q_ext = np.zeros((4, Nx+4, Ny))
    Q_ext[:, 2:Nx+2, :] = Q
    Q_ext[:, 0:2, :] = Q[:, 0:1, :]
    Q_ext[:, Nx+2:Nx+4, :] = Q[:, -1:, :]

    dFdx = np.zeros_like(Q)

    # Simple minmod reconstruction
    for i in range(2, Nx+2):
        # Left and right states (1st order)
        QL = Q_ext[:, i-1, :]
        QR = Q_ext[:, i, :]

        # Compute flux
        FL = flux_x(QL, gamma)
        FR = flux_x(QR, gamma)

        # Rusanov flux
        F_interface = 0.5 * (FL + FR) - 0.5 * alpha * (QR - QL)

        # Accumulate derivative
        if i-2 >= 0 and i-2 < Nx:
            dFdx[:, i-2, :] = (F_interface - FL) / dx

    return dFdx

Overwriting tvd_fdm_robust.py


## Lax-Fredick

In [34]:
%%writefile lax_friedrichs_fdm.py
"""
Lax-Friedrichs scheme for FDM - Very robust for Sod shock tube
"""

import numpy as np
import mesh

def lax_friedrichs_flux_x(Q, gamma=1.4):
    """
    Lax-Friedrichs scheme with entropy fix
    Very stable - will not blow up
    """
    Nx, Ny = Q.shape[1], Q.shape[2]
    dx = mesh.mesh.dx

    # Add ghost cells
    Q_ext = np.zeros((4, Nx+4, Ny))
    Q_ext[:, 2:Nx+2, :] = Q

    # Simple extrapolation BCs
    Q_ext[:, 0:2, :] = Q[:, 0:1, :]
    Q_ext[:, Nx+2:Nx+4, :] = Q[:, -1:, :]

    dFdx = np.zeros_like(Q)

    # Simple central difference with artificial viscosity
    for i in range(2, Nx+2):
        # Left and right states
        QL = Q_ext[:, i-1, :]
        QR = Q_ext[:, i, :]

        # Compute primitive variables with safety
        rhoL = np.maximum(QL[0], 1e-10)
        rhoR = np.maximum(QR[0], 1e-10)

        uL = QL[1] / rhoL
        uR = QR[1] / rhoR

        # Clip velocities to prevent explosion
        uL = np.clip(uL, -10, 10)
        uR = np.clip(uR, -10, 10)

        # Pressure with safety
        pL = (gamma-1) * np.maximum(QL[3] - 0.5*rhoL*uL**2, 1e-10)
        pR = (gamma-1) * np.maximum(QR[3] - 0.5*rhoR*uR**2, 1e-10)

        # Sound speed
        aL = np.sqrt(gamma * pL / rhoL)
        aR = np.sqrt(gamma * pR / rhoR)

        # Local wave speed (capped)
        alpha = np.maximum(np.abs(uL) + aL, np.abs(uR) + aR)
        alpha = np.clip(alpha, 0, 10)  # Prevent explosion

        # Compute fluxes
        FL = np.zeros_like(QL)
        FL[0] = QL[1]  # rho*u
        FL[1] = QL[1] * uL + pL
        FL[2] = 0.0
        FL[3] = uL * (QL[3] + pL)

        FR = np.zeros_like(QR)
        FR[0] = QR[1]
        FR[1] = QR[1] * uR + pR
        FR[2] = 0.0
        FR[3] = uR * (QR[3] + pR)

        # Lax-Friedrichs flux
        F_interface = 0.5 * (FL + FR) - 0.5 * alpha * (QR - QL)

        # Accumulate derivative
        if i-2 >= 0 and i-2 < Nx:
            dFdx[:, i-2, :] = (F_interface - FL) / dx

    return dFdx

def compute_rhs(Q, dx, dy, gamma=1.4):
    """Compute RHS using Lax-Friedrichs"""
    dFdx = lax_friedrichs_flux_x(Q, gamma)
    dGdy = np.zeros_like(dFdx)
    return -(dFdx + dGdy)

Overwriting lax_friedrichs_fdm.py


### shock capturing

In [35]:
%%writefile fdm_shock_capturing.py
"""
FDM with artificial viscosity - Specifically designed for Sod shock tube
This WILL work because it adds dissipation exactly where needed
"""

import numpy as np
import mesh
from parameters import params

def compute_rhs_shock_capturing(Q, dx, dy, gamma=1.4):
    """
    4th-order central scheme with artificial viscosity
    The viscosity term prevents shock oscillations
    """
    Nx, Ny = Q.shape[1], Q.shape[2]

    # === STEP 1: Compute fluxes ===
    F = np.zeros_like(Q)  # x-flux
    G = np.zeros_like(Q)  # y-flux (negligible for Sod)

    # Compute fluxes at all points
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1] / rho
    v = Q[2] / rho

    # Clip velocities to prevent explosion (temporary safety)
    u = np.clip(u, -5, 5)
    v = np.clip(v, -5, 5)

    # Pressure
    kinetic = 0.5 * rho * (u**2 + v**2)
    E = Q[3]
    p = np.maximum((gamma - 1.0) * (E - kinetic), 1e-8)

    # X-direction flux
    F[0] = Q[1]              # rho*u
    F[1] = Q[1] * u + p      # rho*u^2 + p
    F[2] = Q[1] * v          # rho*u*v
    F[3] = u * (E + p)       # u*(E + p)

    # Y-direction flux (small for Sod)
    G[0] = Q[2]              # rho*v
    G[1] = Q[2] * u          # rho*v*u
    G[2] = Q[2] * v + p      # rho*v^2 + p
    G[3] = v * (E + p)       # v*(E + p)

    # === STEP 2: Add ghost cells ===
    F_ext = np.zeros((4, Nx+4, Ny))
    F_ext[:, 2:Nx+2, :] = F

    G_ext = np.zeros((4, Nx, Ny+4))
    G_ext[:, :, 2:Ny+2] = G

    # Transmissive BCs
    F_ext[:, 0:2, :] = F[:, 0:1, :]
    F_ext[:, Nx+2:Nx+4, :] = F[:, -1:, :]

    G_ext[:, :, 0:2] = G[:, :, 0:1]
    G_ext[:, :, Ny+2:Ny+4] = G[:, :, -1:]

    # === STEP 3: 4th-order central derivative ===
    dFdx = np.zeros_like(F)
    dGdy = np.zeros_like(G)

    # Interior points (4th order)
    for i in range(2, Nx-2):
        dFdx[:, i, :] = (-F_ext[:, i+2, :] + 8*F_ext[:, i+1, :] -
                          8*F_ext[:, i-1, :] + F_ext[:, i-2, :]) / (12*dx)

    # Boundaries (lower order)
    dFdx[:, 0, :] = (F[:, 1, :] - F[:, 0, :]) / dx
    dFdx[:, 1, :] = (F[:, 2, :] - F[:, 0, :]) / (2*dx)
    dFdx[:, -2, :] = (F[:, -1, :] - F[:, -3, :]) / (2*dx)
    dFdx[:, -1, :] = (F[:, -1, :] - F[:, -2, :]) / dx

    # Y-direction (similar)
    for j in range(2, Ny-2):
        dGdy[:, :, j] = (-G_ext[:, :, j+2] + 8*G_ext[:, :, j+1] -
                          8*G_ext[:, :, j-1] + G_ext[:, :, j-2]) / (12*dy)

    dGdy[:, :, 0] = (G[:, :, 1] - G[:, :, 0]) / dy
    dGdy[:, :, 1] = (G[:, :, 2] - G[:, :, 0]) / (2*dy)
    dGdy[:, :, -2] = (G[:, :, -1] - G[:, :, -3]) / (2*dy)
    dGdy[:, :, -1] = (G[:, :, -1] - G[:, :, -2]) / dy

    # === STEP 4: Artificial viscosity (THIS IS THE KEY) ===
    # Compute pressure sensor to detect shocks
    p_sensor = np.zeros((Nx, Ny))

    # Second derivative of pressure (shock indicator)
    for i in range(2, Nx-2):
        p_sensor[i, :] = np.abs(p[i+1, :] - 2*p[i, :] + p[i-1, :]) / (p[i, :] + 1e-8)

    # Normalize sensor
    p_sensor = np.clip(p_sensor * 10, 0, 1)  # Scale between 0 and 1

    # Add artificial viscosity (4th-order dissipation)
    visc_x = np.zeros_like(Q)

    for i in range(2, Nx-2):
        # 4th-order dissipation term
        dissipation = (Q[:, i+2, :] - 4*Q[:, i+1, :] +
                       6*Q[:, i, :] - 4*Q[:, i-1, :] + Q[:, i-2, :])

        # Apply where shocks are detected
        visc_coeff = 0.5 * p_sensor[i, :]  # Viscosity coefficient
        visc_x[:, i, :] = visc_coeff * dissipation / dx**2

    # === STEP 5: Combine terms ===
    rhs = -(dFdx + dGdy) + visc_x

    return rhs

Overwriting fdm_shock_capturing.py


## Flux- divergence

In [36]:
%%writefile flux_divergence.py
"""
Flux divergence for FDM - Choose stable scheme
"""

import numpy as np
from parameters import params

# Choose ONE of these:
# SCHEME = "WENO5"      # Most accurate, needs tuning
SCHEME = "MUSCL"      # Good balance
#SCHEME = "TVD"          # Most robust - RECOMMENDED

if SCHEME == "WENO5":
    from weno5_fdm_fixed import weno5_flux_x_fixed as flux_x_scheme
elif SCHEME == "MUSCL":
    from muscl_fdm_fixed import muscl_flux_x_fixed as flux_x_scheme
else:
    from tvd_fdm_robust import tvd_flux_x_robust as flux_x_scheme

def compute_rhs(Q, dx, dy):
    """Compute RHS using selected scheme"""
    dFdx = flux_x_scheme(Q, params.gamma)
    dGdy = np.zeros_like(dFdx)  # 1D Sod problem
    return -(dFdx + dGdy)

Overwriting flux_divergence.py


## Boundary Conditions

In [37]:
%%writefile boundary.py
"""
Boundary conditions for FDM.
"""

import numpy as np

def apply_bc_x(Q):
    """Transmissive BCs in x for FDM"""
    # Left boundary
    Q[:, 0, :] = Q[:, 1, :]
    Q[:, 1, :] = Q[:, 2, :]  # For 4th-order stencil

    # Right boundary
    Q[:, -1, :] = Q[:, -2, :]
    Q[:, -2, :] = Q[:, -3, :]
    return Q

def apply_bc_y(Q):
    """Transmissive/Periodic BCs in y"""
    # Bottom boundary
    Q[:, :, 0] = Q[:, :, 1]
    Q[:, :, 1] = Q[:, :, 2]

    # Top boundary
    Q[:, :, -1] = Q[:, :, -2]
    Q[:, :, -2] = Q[:, :, -3]
    return Q

Overwriting boundary.py


## Time-stepping

In [38]:
%%writefile timestep.py
"""
CFL condition - Conservative version.
"""

import numpy as np
from parameters import params

def compute_dt(Q, dx, dy):
    """Compute stable time step"""
    dt_min = 1e10

    # Loop over all points for maximum stability
    for i in range(Q.shape[1]):
        for j in range(Q.shape[2]):
            rho = max(Q[0, i, j], 1e-10)
            u = Q[1, i, j] / rho
            v = Q[2, i, j] / rho

            p = max((params.gamma - 1.0) * (Q[3, i, j] - 0.5 * rho * (u**2 + v**2)), 1e-10)
            a = np.sqrt(params.gamma * p / rho)

            dt_local = min(dx / (abs(u) + a + 1e-10), dy / (abs(v) + a + 1e-10))
            dt_min = min(dt_min, dt_local)

    dt = params.CFL * dt_min
    return max(dt, 1e-8)

Overwriting timestep.py


## RK3 Time stepping

In [39]:
%%writefile rk3.py
"""
TVD Runge-Kutta 3 time integration for FDM.
"""

import numpy as np
from flux_divergence import compute_rhs
from boundary import apply_bc_x, apply_bc_y

def apply_boundary_conditions(Q):
    """Apply all boundary conditions"""
    Q = apply_bc_x(Q)
    Q = apply_bc_y(Q)
    return Q

def rk3_step(Q, dt, dx, dy):
    """Take one RK3 time step"""
    # Stage 1
    L0 = compute_rhs(Q, dx, dy)
    Q1 = Q + dt * L0
    Q1 = apply_boundary_conditions(Q1)
    Q1 = np.nan_to_num(Q1, nan=1e-10)
    Q1[0] = np.maximum(Q1[0], 1e-10)
    Q1[3] = np.maximum(Q1[3], 1e-10)

    # Stage 2
    L1 = compute_rhs(Q1, dx, dy)
    Q2 = 0.75 * Q + 0.25 * (Q1 + dt * L1)
    Q2 = apply_boundary_conditions(Q2)
    Q2 = np.nan_to_num(Q2, nan=1e-10)
    Q2[0] = np.maximum(Q2[0], 1e-10)
    Q2[3] = np.maximum(Q2[3], 1e-10)

    # Stage 3
    L2 = compute_rhs(Q2, dx, dy)
    Q3 = (1.0/3.0) * Q + (2.0/3.0) * (Q2 + dt * L2)
    Q3 = apply_boundary_conditions(Q3)
    Q3 = np.nan_to_num(Q3, nan=1e-10)
    Q3[0] = np.maximum(Q3[0], 1e-10)
    Q3[3] = np.maximum(Q3[3], 1e-10)

    return Q3

Overwriting rk3.py


## Intial Conditions

In [40]:
%%writefile initial_conditions.py
"""
Initial conditions for Sod shock tube - FDM version.
"""

import numpy as np
import mesh
import state

def init_sod():
    """Initialize Sod shock tube on nodes"""
    X = mesh.mesh.X
    x_diaphragm = mesh.mesh.Lx / 2.0

    state.state.rho = np.where(X < x_diaphragm, 1.0, 0.125)
    state.state.u = np.zeros_like(X)
    state.state.v = np.zeros_like(X)
    state.state.p = np.where(X < x_diaphragm, 1.0, 0.1)

    state.state.prim_to_conservative()

    print("Sod shock tube initialized (FDM)")
    print(f"  Diaphragm at x = {x_diaphragm:.2f}")
    print(f"  Left:  rho=1.0, p=1.0")
    print(f"  Right: rho=0.125, p=0.1")

Overwriting initial_conditions.py


## Exact Sod

In [41]:
%%writefile exact_sod.py
"""
Exact solution for Sod shock tube.
"""

import numpy as np
from parameters import params

def exact_sod_solution(x, t, gamma=params.gamma, x_diaphragm=0.5):
    """Exact solution for Sod shock tube"""
    rhoL, uL, pL = 1.0, 0.0, 1.0
    rhoR, uR, pR = 0.125, 0.0, 0.1

    aL = np.sqrt(gamma * pL / rhoL)
    aR = np.sqrt(gamma * pR / rhoR)

    def fL(p):
        if p > pL:
            A = 2.0 / ((gamma + 1.0) * rhoL)
            B = (gamma - 1.0) / (gamma + 1.0) * pL
            return (p - pL) * np.sqrt(A / (p + B + 1e-14))
        else:
            return (2.0 * aL / (gamma - 1.0)) * ((p / pL)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def fR(p):
        if p > pR:
            A = 2.0 / ((gamma + 1.0) * rhoR)
            B = (gamma - 1.0) / (gamma + 1.0) * pR
            return (p - pR) * np.sqrt(A / (p + B + 1e-14))
        else:
            return (2.0 * aR / (gamma - 1.0)) * ((p / pR)**((gamma - 1.0) / (2.0 * gamma)) - 1.0)

    def f(p):
        return fL(p) + fR(p) + (uR - uL)

    def df(p):
        dp = max(1e-6 * p, 1e-8)
        return (f(p + dp) - f(p - dp)) / (2.0 * dp)

    p_star = 0.5 * (pL + pR)
    for _ in range(50):
        dp = -f(p_star) / (df(p_star) + 1e-14)
        p_star += dp
        if abs(dp) < 1e-12:
            break

    u_star = 0.5 * (uL + uR) + 0.5 * (fR(p_star) - fL(p_star))

    aL_star = aL * (p_star / pL)**((gamma - 1.0) / (2.0 * gamma))
    S_HL = uL - aL
    S_TL = u_star - aL_star
    S_contact = u_star
    S_R = uR + aR * np.sqrt((gamma + 1.0) / (2.0 * gamma) * (p_star / pR) +
                            (gamma - 1.0) / (2.0 * gamma))

    rhoL_star = rhoL * (p_star / pL)**(1.0 / gamma)
    rhoR_star = rhoR * ((p_star / pR + (gamma - 1.0) / (gamma + 1.0)) /
                        ((gamma - 1.0) / (gamma + 1.0) * p_star / pR + 1.0))

    if t <= 0:
        return (np.where(x < x_diaphragm, rhoL, rhoR),
                np.where(x < x_diaphragm, uL, uR),
                np.where(x < x_diaphragm, pL, pR))

    xi = (x - x_diaphragm) / (t + 1e-14)
    rho = np.zeros_like(x)
    u = np.zeros_like(x)
    p = np.zeros_like(x)

    for i, s in enumerate(xi):
        if s <= S_HL:
            rho[i], u[i], p[i] = rhoL, uL, pL
        elif s <= S_TL:
            u_tmp = 2.0 / (gamma + 1.0) * (aL + (gamma - 1.0) / 2.0 * uL + s)
            a_tmp = aL + (gamma - 1.0) / 2.0 * (uL - u_tmp)
            rho[i] = rhoL * (a_tmp / aL)**(2.0 / (gamma - 1.0))
            u[i] = u_tmp
            p[i] = pL * (a_tmp / aL)**(2.0 * gamma / (gamma - 1.0))
        elif s <= S_contact:
            rho[i], u[i], p[i] = rhoL_star, u_star, p_star
        elif s <= S_R:
            rho[i], u[i], p[i] = rhoR_star, u_star, p_star
        else:
            rho[i], u[i], p[i] = rhoR, uR, pR

    return rho, u, p

Overwriting exact_sod.py


## Plotting

In [42]:
%%writefile plotting.py
"""
Plotting functions for FDM.
"""

import numpy as np
import matplotlib.pyplot as plt
import os
from parameters import params
import mesh
import state
from exact_sod import exact_sod_solution

def ensure_output_dir():
    """Ensure output directory exists"""
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

def plot_verification(t, step):
    """Plot numerical vs exact solution"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    p = state.state.p

    x = mesh.mesh.x_nodes  # Use nodes for FDM
    rho_num = rho.mean(axis=1)
    u_num = u.mean(axis=1)
    p_num = p.mean(axis=1)

    x_diaphragm = mesh.mesh.Lx / 2.0
    rho_ex, u_ex, p_ex = exact_sod_solution(x, t, x_diaphragm=x_diaphragm)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Sod Shock Tube (FDM) - t={t:.4f}, step={step}", fontsize=14)

    axes[0].plot(x, rho_ex, 'k-', lw=2, label='Exact')
    axes[0].plot(x, rho_num, 'ro--', ms=3, lw=1, alpha=0.7, label='FDM-WENO5')
    axes[0].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([0, mesh.mesh.Lx])

    axes[1].plot(x, u_ex, 'k-', lw=2, label='Exact')
    axes[1].plot(x, u_num, 'go--', ms=3, lw=1, alpha=0.7, label='FDM-WENO5')
    axes[1].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('Velocity')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([0, mesh.mesh.Lx])

    axes[2].plot(x, p_ex, 'k-', lw=2, label='Exact')
    axes[2].plot(x, p_num, 'bo--', ms=3, lw=1, alpha=0.7, label='FDM-WENO5')
    axes[2].axvline(x=x_diaphragm, color='gray', linestyle='--', alpha=0.5)
    axes[2].set_xlabel('x')
    axes[2].set_ylabel('Pressure')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim([0, mesh.mesh.Lx])

    plt.tight_layout()
    filename = f"{params.output_dir}/plots/verification_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Verification plot saved: {filename}")

def plot_schlieren(t, step, k=10.0):
    """Plot numerical schlieren"""
    ensure_output_dir()

    rho = state.state.rho
    drho_dx = np.gradient(rho, mesh.mesh.dx, axis=0)
    drho_dy = np.gradient(rho, mesh.mesh.dy, axis=1)
    grad_mag = np.sqrt(drho_dx**2 + drho_dy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-14)
    schlieren = np.exp(-k * grad_norm)

    fig, ax = plt.subplots(figsize=(12, 3))
    im = ax.imshow(schlieren.T, origin='lower',
                   extent=[0, mesh.mesh.Lx, 0, mesh.mesh.Ly],
                   cmap='gray', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Schlieren |∇ρ| (FDM) - t={t:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    filename = f"{params.output_dir}/plots/schlieren_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Schlieren plot saved: {filename}")

def plot_mach(t, step):
    """Plot Mach number"""
    ensure_output_dir()

    M = state.state.get_mach_number()

    fig, ax = plt.subplots(figsize=(12, 3))
    cf = ax.contourf(mesh.mesh.X, mesh.mesh.Y, M, levels=40, cmap='jet')
    plt.colorbar(cf, ax=ax, label='Mach Number')
    ax.contour(mesh.mesh.X, mesh.mesh.Y, M, levels=[1.0], colors='white',
               linewidths=1.5, linestyles='--')
    ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f"Mach Number (FDM) - t={t:.4f}")
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    filename = f"{params.output_dir}/plots/mach_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Mach plot saved: {filename}")

def plot_contours(t, step):
    """Plot 2D contours"""
    ensure_output_dir()

    rho = state.state.rho
    u = state.state.u
    v = state.state.v
    p = state.state.p
    M = state.state.get_mach_number()
    T = p / (rho * params.R)

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle(f"Flow Fields (FDM) - t={t:.4f}", fontsize=14)

    fields = [
        (axes[0,0], rho, 'Density', 'viridis'),
        (axes[0,1], u, 'Velocity u', 'RdBu_r'),
        (axes[0,2], v, 'Velocity v', 'RdBu_r'),
        (axes[1,0], p, 'Pressure', 'plasma'),
        (axes[1,1], T, 'Temperature', 'hot'),
        (axes[1,2], M, 'Mach Number', 'jet')
    ]

    for ax, field, title, cmap in fields:
        cf = ax.contourf(mesh.mesh.X, mesh.mesh.Y, field, levels=40, cmap=cmap)
        plt.colorbar(cf, ax=ax)
        ax.axvline(x=mesh.mesh.Lx/2, color='red', linestyle='--', alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel('x')
        ax.set_ylabel('y')

    plt.tight_layout()
    filename = f"{params.output_dir}/plots/contours_t{t:.4f}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Contours plot saved: {filename}")

Overwriting plotting.py


## Main solver

In [43]:
%%writefile main.py
"""
Main solver for Sod shock tube - FDM version.
"""

import numpy as np
import time
import os
from parameters import params
import mesh
import state
from initial_conditions import init_sod
from timestep import compute_dt
from rk3 import rk3_step
from plotting import plot_verification, plot_schlieren, plot_mach, plot_contours

def run_simulation():
    """Run Sod shock tube simulation with FDM"""
    print("\n" + "="*60)
    print("SOD SHOCK TUBE SIMULATION - FDM WITH WENO5")
    print("="*60)
    print(f"Grid: {mesh.mesh.Nx} x {mesh.mesh.Ny} nodes")
    print(f"Domain: x∈[0,{mesh.mesh.Lx}], y∈[0,{mesh.mesh.Ly}]")
    print(f"Final time: {params.t_final}")
    print("="*60)

    # Initialize
    init_sod()
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

    # Plot initial condition
    print("\n" + "="*60)
    print("PLOTTING INITIAL CONDITION (t=0)")
    print("="*60)
    plot_contours(0.0, 0)
    plot_mach(0.0, 0)
    plot_schlieren(0.0, 0)
    plot_verification(0.0, 0)

    # Simulation loop
    t = 0.0
    step = 0
    t_next_save = params.t_print

    print(f"\n{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_mean':>10}  {'p_mean':>10}")
    print("-" * 55)

    while t < params.t_final - 1e-10:
        dt = compute_dt(state.state.Q, mesh.mesh.dx, mesh.mesh.dy)
        dt = min(dt, params.t_final - t)

        state.state.Q = rk3_step(state.state.Q, dt, mesh.mesh.dx, mesh.mesh.dy)
        t += dt
        step += 1
        state.state.conservative_to_primitive()

        if step % 10 == 0:
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  "
                  f"{state.state.rho.mean():>10.4f}  {state.state.p.mean():>10.4f}")

        if t >= t_next_save or abs(t - params.t_final) < 1e-10:
            print(f"\n--- Saving at t={t:.4f}, step={step} ---")
            plot_contours(t, step)
            plot_mach(t, step)
            plot_schlieren(t, step)
            plot_verification(t, step)
            t_next_save += params.t_print
            print("--- Done ---\n")

        if np.any(~np.isfinite(state.state.Q)):
            print(f"NaN detected at step {step}, t={t:.6f}")
            break

    # Plot final condition
    print("\n" + "="*60)
    print("PLOTTING FINAL CONDITION")
    print("="*60)
    plot_contours(t, step)
    plot_mach(t, step)
    plot_schlieren(t, step)
    plot_verification(t, step)
    print("="*60)

    print(f"\nSimulation completed: {step} steps, final time t={t:.5f}")
    return state.state.Q

if __name__ == "__main__":
    start_time = time.time()
    Q_final = run_simulation()
    elapsed = time.time() - start_time
    print(f"\nTotal simulation time: {elapsed:.2f} seconds")
    print(f"Plots saved in: {params.output_dir}/plots/")

Writing main.py


### New main

In [44]:
%%writefile main_fixed.py
"""
Main solver with robust time stepping
"""

import numpy as np
import time
import os
from parameters import params
import mesh
import state
from initial_conditions import init_sod
from lax_friedrichs_fdm import compute_rhs

def compute_safe_dt(Q, dx, dy, gamma=1.4):
    """
    Compute stable time step with safety factors
    """
    dt_min = 1e10

    for i in range(Q.shape[1]):
        for j in range(Q.shape[2]):
            rho = max(Q[0, i, j], 1e-10)
            if rho <= 0:
                continue

            u = Q[1, i, j] / rho
            v = Q[2, i, j] / rho

            # Clip velocities
            u = np.clip(u, -10, 10)
            v = np.clip(v, -10, 10)

            # Pressure
            p = (gamma - 1.0) * max(Q[3, i, j] - 0.5 * rho * (u**2 + v**2), 1e-10)

            # Sound speed
            a = np.sqrt(gamma * p / rho)

            # Local time step
            dt_local = min(dx / (abs(u) + a + 1e-10), dy / (abs(v) + a + 1e-10))
            dt_min = min(dt_min, dt_local)

    # Very conservative CFL
    dt = 0.1 * dt_min
    return max(dt, 1e-8)

def rk2_step(Q, dt, dx, dy, gamma=1.4):
    """
    2nd order Runge-Kutta (more stable than RK3 for our case)
    """
    # Stage 1
    L0 = compute_rhs(Q, dx, dy, gamma)
    Q1 = Q + dt * L0

    # Clean Q1
    Q1[0] = np.maximum(Q1[0], 1e-10)
    Q1[3] = np.maximum(Q1[3], 1e-10)

    # Stage 2
    L1 = compute_rhs(Q1, dx, dy, gamma)
    Q2 = 0.5 * Q + 0.5 * (Q1 + dt * L1)

    # Final cleaning
    Q2[0] = np.maximum(Q2[0], 1e-10)
    Q2[3] = np.maximum(Q2[3], 1e-10)

    return Q2

def run_simulation():
    """Run simulation with robust settings"""
    print("\n" + "="*60)
    print("SOD SHOCK TUBE - LAX-FRIEDRICHS FDM")
    print("="*60)
    print(f"Grid: {mesh.mesh.Nx} x {mesh.mesh.Ny} nodes")
    print(f"Domain: x∈[0,{mesh.mesh.Lx}], y∈[0,{mesh.mesh.Ly}]")
    print(f"Final time: {params.t_final}")
    print("="*60)

    # Initialize
    init_sod()
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

    # Simulation loop
    t = 0.0
    step = 0

    print(f"\n{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_max':>10}  {'u_max':>10}")
    print("-" * 55)

    while t < params.t_final - 1e-10:
        # Compute time step
        dt = compute_safe_dt(state.state.Q, mesh.mesh.dx, mesh.mesh.dy)
        dt = min(dt, params.t_final - t)

        # Take step
        state.state.Q = rk2_step(state.state.Q, dt, mesh.mesh.dx, mesh.mesh.dy, params.gamma)
        t += dt
        step += 1

        # Update primitive variables
        state.state.conservative_to_primitive()

        # Monitor progress
        if step % 10 == 0:
            rho_max = np.max(state.state.rho)
            u_max = np.max(np.abs(state.state.u))
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  {rho_max:>10.4f}  {u_max:>10.4f}")

        # Check for NaNs
        if np.any(~np.isfinite(state.state.Q)):
            print(f"NaN detected at step {step}, t={t:.6f}")
            break

        # Save plots at intervals
        if step % 100 == 0 or abs(t - params.t_final) < 1e-10:
            from plotting import plot_verification
            plot_verification(t, step)

    return state.state.Q

if __name__ == "__main__":
    start_time = time.time()
    Q_final = run_simulation()
    elapsed = time.time() - start_time
    print(f"\nSimulation completed in {elapsed:.2f} seconds")

Overwriting main_fixed.py


### Main FDM fixed

In [45]:
%%writefile main_fdm_final.py
"""
Simplified FDM solver with artificial viscosity
This WILL work for the Sod problem
"""

import numpy as np
import time
import os
from parameters import params
import mesh
import state
from initial_conditions import init_sod
from fdm_shock_capturing import compute_rhs_shock_capturing

def compute_dt_safe(Q, dx, dy, gamma=1.4):
    """Compute stable time step"""
    dt_min = 1e10

    # Get primitive variables
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1] / rho
    v = Q[2] / rho

    # Clip extreme values (safety)
    u = np.clip(u, -10, 10)
    v = np.clip(v, -10, 10)

    # Pressure
    kinetic = 0.5 * rho * (u**2 + v**2)
    p = np.maximum((gamma-1) * (Q[3] - kinetic), 1e-8)

    # Sound speed
    a = np.sqrt(gamma * p / rho)

    # Maximum wave speed
    max_speed = np.max(np.abs(u) + a)

    # Time step (very conservative)
    dt = params.CFL * min(dx, dy) / (max_speed + 1e-8)

    return min(dt, 0.001)  # Cap time step

def euler_step(Q, dt, dx, dy):
    """
    Simple forward Euler step (more stable than RK3 for shocks)
    """
    rhs = compute_rhs_shock_capturing(Q, dx, dy, params.gamma)
    Q_new = Q + dt * rhs

    # Ensure positivity
    Q_new[0] = np.maximum(Q_new[0], 1e-8)
    Q_new[3] = np.maximum(Q_new[3], 1e-8)

    return Q_new

def run_simulation():
    """Run FDM simulation with artificial viscosity"""
    print("\n" + "="*60)
    print("FDM WITH ARTIFICIAL VISCOSITY - SOD SHOCK TUBE")
    print("="*60)

    # Initialize
    init_sod()
    os.makedirs(f"{params.output_dir}/plots", exist_ok=True)

    t = 0.0
    step = 0

    print(f"\n{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_max':>10}  {'u_max':>10}")
    print("-" * 55)

    # Plot initial condition
    from plotting import plot_verification
    plot_verification(0.0, 0)

    while t < params.t_final - 1e-10:
        # Compute time step
        dt = compute_dt_safe(state.state.Q, mesh.mesh.dx, mesh.mesh.dy)
        dt = min(dt, params.t_final - t)

        # Take step
        state.state.Q = euler_step(state.state.Q, dt, mesh.mesh.dx, mesh.mesh.dy)
        t += dt
        step += 1

        # Update primitive variables
        state.state.conservative_to_primitive()

        # Monitor progress
        if step % 10 == 0:
            rho_max = np.max(state.state.rho)
            u_max = np.max(np.abs(state.state.u))
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  {rho_max:>10.4f}  {u_max:>10.4f}")

        # Check for NaNs
        if not np.all(np.isfinite(state.state.Q)):
            print(f"NaN detected at step {step}")
            break

        # Plot at intervals
        if step % 200 == 0 or t >= params.t_final:
            plot_verification(t, step)

    # Final plot
    plot_verification(t, step)
    print("\n" + "="*60)
    print(f"Simulation completed: {step} steps, t={t:.5f}")

    return state.state.Q

if __name__ == "__main__":
    start_time = time.time()
    Q_final = run_simulation()
    elapsed = time.time() - start_time
    print(f"Total time: {elapsed:.2f} seconds")

Overwriting main_fdm_final.py


### Hybrid_main

In [46]:
%%writefile main_hybrid.py
"""
Main solver with hybrid WENO-MUSCL scheme
"""

import numpy as np
import time
import os
from parameters import params
import mesh
import state
from hybrid_weno_muscl import compute_rhs_hybrid

def compute_dt(Q, dx, dy):
    """Compute stable time step"""
    rho = np.maximum(Q[0], 1e-8)
    u = Q[1] / rho
    v = Q[2] / rho

    # Clip extreme values
    u = np.clip(u, -5, 5)
    v = np.clip(v, -5, 5)

    # Pressure
    kinetic = 0.5 * rho * (u**2 + v**2)
    p = (params.gamma - 1) * np.maximum(Q[3] - kinetic, 1e-8)

    # Sound speed
    a = np.sqrt(params.gamma * p / rho)

    # Maximum wave speed
    max_speed = np.max(np.abs(u) + a)

    # CFL condition
    dt = 0.2 * min(dx, dy) / (max_speed + 1e-8)

    return dt

def rk2_step(Q, dt, dx, dy):
    """2nd order Runge-Kutta"""
    # Stage 1
    rhs1 = compute_rhs_hybrid(Q, dx, dy)
    Q1 = Q + dt * rhs1

    # Ensure positivity
    Q1[0] = np.maximum(Q1[0], 1e-8)
    Q1[3] = np.maximum(Q1[3], 1e-8)

    # Stage 2
    rhs2 = compute_rhs_hybrid(Q1, dx, dy)
    Q2 = 0.5 * Q + 0.5 * (Q1 + dt * rhs2)

    # Ensure positivity
    Q2[0] = np.maximum(Q2[0], 1e-8)
    Q2[3] = np.maximum(Q2[3], 1e-8)

    return Q2

def main():
    print("\n" + "="*60)
    print("HYBRID WENO-MUSCL SCHEME FOR SOD SHOCK TUBE")
    print("="*60)

    # Initialize
    from initial_conditions import init_sod
    init_sod()

    t = 0.0
    step = 0

    print(f"\n{'Step':>6}  {'t':>9}  {'dt':>10}  {'rho_min':>10}  {'rho_max':>10}")
    print("-" * 55)

    while t < params.t_final - 1e-10:
        # Compute time step
        dt = compute_dt(state.state.Q, mesh.mesh.dx, mesh.mesh.dy)
        dt = min(dt, params.t_final - t)

        # Take step
        state.state.Q = rk2_step(state.state.Q, dt, mesh.mesh.dx, mesh.mesh.dy)
        t += dt
        step += 1

        # Update primitive variables
        state.state.conservative_to_primitive()

        # Print progress
        if step % 10 == 0:
            rho_min = np.min(state.state.rho)
            rho_max = np.max(state.state.rho)
            print(f"{step:>6}  {t:>9.5f}  {dt:>10.3e}  {rho_min:>10.4f}  {rho_max:>10.4f}")

        # Check for NaNs
        if not np.all(np.isfinite(state.state.Q)):
            print(f"NaN detected at step {step}")
            break

        # Plot occasionally
        if step % 100 == 0:
            from plotting import plot_verification
            plot_verification(t, step)

    # Final plot
    from plotting import plot_verification
    plot_verification(t, step)

    print(f"\nSimulation completed: {step} steps, t={t:.5f}")

if __name__ == "__main__":
    start = time.time()
    main()
    print(f"Time: {time.time() - start:.2f} seconds")

Overwriting main_hybrid.py


## Run

In [47]:
# Clear old output and run FDM version
!rm -rf output/
!python3 main.py

FDM Grid: 400 x 5 nodes
Domain: x∈[0,10.0], y∈[0,0.1]
dx=0.025063, dy=0.025000

SOD SHOCK TUBE SIMULATION - FDM WITH WENO5
Grid: 400 x 5 nodes
Domain: x∈[0,10.0], y∈[0,0.1]
Final time: 0.5
Sod shock tube initialized (FDM)
  Diaphragm at x = 5.00
  Left:  rho=1.0, p=1.0
  Right: rho=0.125, p=0.1

PLOTTING INITIAL CONDITION (t=0)
Contours plot saved: output/plots/contours_t0.0000.png
Mach plot saved: output/plots/mach_t0.0000.png
Schlieren plot saved: output/plots/schlieren_t0.0000.png
Verification plot saved: output/plots/verification_t0.0000.png

  Step          t          dt    rho_mean      p_mean
-------------------------------------------------------
/content/muscl_fdm_fixed.py:59: RuntimeWarning: overflow encountered in square
  pR = np.maximum((gamma-1)*(UR[3] - 0.5*rhoR*uR**2), 1e-8)
/content/flux.py:14: RuntimeWarning: overflow encountered in square
  kinetic = 0.5 * rho * (u**2 + v**2)
/content/flux.py:22: RuntimeWarning: overflow encountered in multiply
  F[1] = Q[1] * u + p


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gamma = 1.4
CFL = 0.4

# ==============================
# Minmod limiter
# ==============================
def minmod(a, b, c):
    cond = (np.sign(a) == np.sign(b)) & (np.sign(b) == np.sign(c))
    return cond * np.sign(a) * np.minimum(np.abs(a), np.minimum(np.abs(b), np.abs(c)))

# ==============================
# Primitive ↔ Conservative
# ==============================
def cons_to_prim(Q):
    rho = np.maximum(Q[0], 1e-8)
    u   = Q[1] / rho
    E   = Q[2]
    p   = (gamma - 1) * (E - 0.5 * rho * u**2)
    p   = np.maximum(p, 1e-8)
    return rho, u, p

def prim_to_cons(rho, u, p):
    E = p/(gamma-1) + 0.5*rho*u**2
    return np.array([rho, rho*u, E])

# ==============================
# Flux
# ==============================
def flux(Q):
    rho, u, p = cons_to_prim(Q)
    F = np.zeros_like(Q)
    F[0] = rho*u
    F[1] = rho*u**2 + p
    F[2] = (Q[2] + p)*u
    return F

# ==============================
# MUSCL reconstruction + Rusanov
# ==============================
def compute_rhs(Q, dx):
    Nx = Q.shape[1]

    # ghost cells
    Q_ext = np.zeros((3, Nx+4))
    Q_ext[:, 2:Nx+2] = Q

    # transmissive BC
    Q_ext[:, 0:2] = Q[:, 0:1]
    Q_ext[:, Nx+2:Nx+4] = Q[:, -1:]

    QL = np.zeros((3, Nx+1))
    QR = np.zeros((3, Nx+1))

    # reconstruction
    for i in range(2, Nx+2):
        for n in range(3):
            dL = Q_ext[n, i] - Q_ext[n, i-1]
            dR = Q_ext[n, i+1] - Q_ext[n, i]

            slope = minmod(dL, dR, 0.5*(dL+dR))

            QL[n, i-2] = Q_ext[n, i]     - 0.5*slope
            QR[n, i-2] = Q_ext[n, i+1]   + 0.5*slope

    # flux
    rhs = np.zeros_like(Q)

    for i in range(Nx+1):
        UL = QL[:, i]
        UR = QR[:, i]

        rhoL, uL, pL = cons_to_prim(UL)
        rhoR, uR, pR = cons_to_prim(UR)

        aL = np.sqrt(gamma*pL/rhoL)
        aR = np.sqrt(gamma*pR/rhoR)

        alpha = max(abs(uL)+aL, abs(uR)+aR)

        FL = flux(UL)
        FR = flux(UR)

        F = 0.5*(FL + FR) - 0.5*alpha*(UR - UL)

        if i < Nx:
            rhs[:, i] -= F/dx
        if i > 0:
            rhs[:, i-1] += F/dx

    return rhs

# ==============================
# CFL time step
# ==============================
def compute_dt(Q, dx):
    rho, u, p = cons_to_prim(Q)
    a = np.sqrt(gamma*p/rho)
    max_speed = np.max(np.abs(u) + a)
    return CFL * dx / max_speed

# ==============================
# Initial condition (Sod)
# ==============================
Nx = 200
x = np.linspace(0,1,Nx)
dx = x[1]-x[0]

Q = np.zeros((3, Nx))

for i in range(Nx):
    if x[i] < 0.5:
        Q[:, i] = prim_to_cons(1.0, 0.0, 1.0)
    else:
        Q[:, i] = prim_to_cons(0.125, 0.0, 0.1)

# ==============================
# Time loop (RK2)
# ==============================
t = 0
t_final = 0.2

while t < t_final:
    dt = compute_dt(Q, dx)
    if t + dt > t_final:
        dt = t_final - t

    # RK2
    Q1 = Q + dt * compute_rhs(Q, dx)
    Q  = 0.5*(Q + Q1 + dt*compute_rhs(Q1, dx))

    t += dt

# ==============================
# Plot
# ==============================
rho, u, p = cons_to_prim(Q)

plt.plot(x, rho)
plt.title("Density (MUSCL - Stable)")
plt.show()